# Manual shot rejection of healthy volunteers

Use the **CarDpy3** Python kernel. Run the configuration and inventory cells, then the review cell.

This loads `01_Unrung/Unrung.nii` (Gibbs-corrected data), registers the interior slices using CarDpy's 2D affine registration, and exports the exact full-resolution magnitude images supplied to the reviewer. The reviewer reads this saved NIfTI. Zoom and intensity controls affect only the display. This matches the registration-before-rejection order in the CarDpy pipeline; registration is recomputed from Unrung data, so it may differ from older processing runs.

Both `Blip_Up` and `Blip_Down` are included by default. First and last slices are excluded. Every shot begins as KEEP; no automated rejection is run and no automated decisions are shown.


In [1]:
from pathlib import Path
from cardpy.manual_rejection import run

ROOT = Path('/Volumes/T7/Smart_Health_Reorganized/01_Processed_Data')
OUTPUT = Path('/Volumes/T7/Smart_Health_Reorganized/02_Manual_Shot_Rejection')
SERIES = ['Blip_Up']  # Or select only one direction.
VOLUNTEERS = None  # All volunteers; or ['Healthy_Volunteer_006'].
ALGORITHM = 'Affine'


In [2]:
# Inventory only: does not register, open windows, or write files.
run(ROOT, OUTPUT, series=SERIES, volunteers=VOLUNTEERS,
    algorithm=ALGORITHM, dry_run=True)


48 series; outputs: /Volumes/T7/Smart_Health_Reorganized/02_Manual_Shot_Rejection
Healthy_Volunteer_006/Blip_Up: slices 2–10, 80 shots per slice
Healthy_Volunteer_007/Blip_Up: slices 2–8, 80 shots per slice
Healthy_Volunteer_008/Blip_Up: slices 2–10, 80 shots per slice
Healthy_Volunteer_009/Blip_Up: slices 2–10, 80 shots per slice
Healthy_Volunteer_010/Blip_Up: slices 2–10, 80 shots per slice
Healthy_Volunteer_011/Blip_Up: slices 2–12, 80 shots per slice
Healthy_Volunteer_012/Blip_Up: slices 2–8, 80 shots per slice
Healthy_Volunteer_013/Blip_Up: slices 2–8, 80 shots per slice
Healthy_Volunteer_014/Blip_Up: slices 2–10, 80 shots per slice
Healthy_Volunteer_015/Blip_Up: slices 2–8, 80 shots per slice
Healthy_Volunteer_016/Blip_Up: slices 2–8, 80 shots per slice
Healthy_Volunteer_017/Blip_Up: slices 2–8, 80 shots per slice
Healthy_Volunteer_018/Blip_Up: slices 2–8, 80 shots per slice
Healthy_Volunteer_019/Blip_Up: slices 2–8, 80 shots per slice
Healthy_Volunteer_020/Blip_Up: slices 2–8, 8

Click a tile to toggle **green = keep / red = reject**. Tile numbers match the exported NIfTI volumes (1-based). Use **Apply slice and continue** or Return to save that slice and advance. **Stop (discard this slice)** or closing the window stops the loop and discards the current slice; earlier applied slices remain saved. Rerun the cell to resume. Finishing a series automatically advances to the next one.


In [ ]:
try:
    run(ROOT, OUTPUT, series=SERIES, volunteers=VOLUNTEERS, algorithm=ALGORITHM)
except (InterruptedError, KeyboardInterrupt):
    print('Stopped. Applied slices are saved; rerun this cell to resume.')


48 series; outputs: /Volumes/T7/Smart_Health_Reorganized/02_Manual_Shot_Rejection
Healthy_Volunteer_006/Blip_Up: registering Gibbs-corrected interior slices...


/opt/anaconda3/envs/cardpy/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Registering 2D data to lowest b-value (0).
Healthy_Volunteer_006/Blip_Up: source slice 2/11
Healthy_Volunteer_006/Blip_Up: source slice 3/11
Healthy_Volunteer_006/Blip_Up: source slice 4/11
Healthy_Volunteer_006/Blip_Up: source slice 5/11
Healthy_Volunteer_006/Blip_Up: source slice 6/11
Healthy_Volunteer_006/Blip_Up: source slice 7/11
Healthy_Volunteer_006/Blip_Up: source slice 8/11
Healthy_Volunteer_006/Blip_Up: source slice 9/11
Healthy_Volunteer_006/Blip_Up: source slice 10/11
Healthy_Volunteer_006/Blip_Up: complete. Decisions: /Volumes/T7/Smart_Health_Reorganized/02_Manual_Shot_Rejection/Healthy_Volunteer_006/Blip_Up/Shot_Decisions.csv
Healthy_Volunteer_007/Blip_Up: registering Gibbs-corrected interior slices...
Registering 2D data to lowest b-value (0).
Healthy_Volunteer_007/Blip_Up: source slice 2/9
Healthy_Volunteer_007/Blip_Up: source slice 3/9
Healthy_Volunteer_007/Blip_Up: source slice 4/9
Healthy_Volunteer_007/Blip_Up: source slice 5/9
Healthy_Volunteer_007/Blip_Up: source s

Each `OUTPUT/Healthy_Volunteer_XXX/Blip_Up/` (or `Blip_Down/`) contains:

- `Review_Input.nii.gz`, `.bvals`, `.bvecs`: registered images actually reviewed, with only interior slices and original source volume order. The affine accounts for the removed first slice. Spatial orientation inherits the source NIfTI.
- `Shot_Decisions.csv`: one keep/reject decision per shot in each completed slice, with original slice/volume indices, displayed tile numbers, b-values, and b-vectors.
- `Review_State.json`: checkpoint with completion flags, keep arrays, source slice mapping, and input/registration settings. Only completed slices have valid decisions. Unreviewed edge slices have no labels.

Source files and existing `02_Rejected` outputs are preserved. The export retains every shot; rejection labels are stored separately, including when all shots in a slice are rejected. No replacement/fallback shots are inserted.

To redo a review or change registration settings, choose a new `OUTPUT` directory. A matching output directory resumes unfinished slices and skips completed series.

Terminal alternative (from the CarDpy repository, in the CarDpy3 environment):

```sh
python -m cardpy.manual_rejection
```

Optional flags: `--dry-run`, `--series Blip_Up`, `--volunteers Healthy_Volunteer_006`, `--output /path/to/new/review`.
